# Shared trajectory memory: retrieval-only and M2

This launcher uses completed **baseline training runs** already on Drive. It builds one successful-trajectory corpus and MPNet index per benchmark. Retrieval-only and action-conditioned M2 use those same artifacts. The existing `m2_launcher.ipynb` and its results stay separate. Run one flagged stage at a time; the GPU smoke and pilot stages can wait until compute is available.


In [ ]:
GITHUB_REPO = 'https://github.com/nave-nap/AgenticSDS.git'
REVISION = 'origin/main'  # Change to the published commit SHA for a frozen run.
DRIVE_ROOT = '/content/drive/MyDrive/agent_support'
BENCHMARKS = ('alfworld', 'textcraft')
CONDITIONS = ('expel_retrieval_only_v1', 'm2_trajectory_v1')
BUILD_TRAJECTORY_MEMORY = False
RUN_TRAJECTORY_SMOKE = False
RUN_TRAJECTORY_PILOT = False
RUN_TRAJECTORY_COMPARE = False
assert GITHUB_REPO.startswith('https://github.com/')


## 1. Set up Drive and the pinned repository

Run this cell after the settings cell. A GPU is needed for smoke and pilot runs. Building memory can run on CPU.


In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib, json, os, subprocess, sys, urllib.request
os.environ['PYTHONHASHSEED'] = '0'
drive.mount('/content/drive')
repo = Path('/content/agent_support_repo')
if not repo.exists():
    subprocess.run(['git', 'clone', GITHUB_REPO, str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', REVISION], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(repo / 'requirements.txt')], check=True)
data_root = Path.home() / '.cache' / 'alfworld'
if not (data_root / 'json_2.1.1').exists():
    subprocess.run(['alfworld-download'], check=True)
os.chdir(repo)
root = Path(DRIVE_ROOT)
root.mkdir(parents=True, exist_ok=True)
commit = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
print('Repository commit:', commit)

def run_visible(command, label):
    completed = subprocess.run(command, cwd=repo, text=True, capture_output=True)
    print(completed.stdout[-5000:], flush=True)
    if completed.returncode:
        print(completed.stderr[-5000:], flush=True)
        raise RuntimeError(f'{label} failed (exit {completed.returncode}); inspect the latest run.json on Drive')


## 2. Build the shared successful-trajectory memory

Set `BUILD_TRAJECTORY_MEMORY=True` in the settings cell, rerun it, and run this cell. It finds each prior full baseline training run by its training-manifest hash. It does not require retraining or a matching old Git commit. It excludes development and held-out tasks by ID and content hash. The index is built with MPNet on CPU.


In [ ]:
from agent_support.common import canonical_digest, read_json

def existing_training_run(benchmark):
    training_manifest = read_json(root / 'manifests' / benchmark / 'training.json')
    expected_hash = canonical_digest(training_manifest)
    candidates = []
    for directory in (root / 'training_runs').glob(f'{benchmark}_*'):
        path = directory / 'run.json'
        if path.exists():
            meta = read_json(path)
            if (meta.get('status') == 'complete' and meta.get('condition') == 'baseline'
                    and meta.get('partition') == 'training' and meta.get('limit') is None
                    and meta.get('selected_manifest_sha256') == expected_hash
                    and meta.get('selected_task_count') == meta.get('total_manifest_task_count')):
                candidates.append(directory)
    if not candidates:
        raise RuntimeError(f'No complete baseline training run with matching manifest for {benchmark}')
    return max(candidates, key=lambda path: path.name)

if BUILD_TRAJECTORY_MEMORY:
    for benchmark in BENCHMARKS:
        folder = root / 'manifests' / benchmark
        command = [sys.executable, '-m', 'agent_support.trajectory_memory',
                   '--run', str(existing_training_run(benchmark)),
                   '--exclude-manifest', str(folder / 'development.json'),
                   '--exclude-manifest', str(folder / 'heldout.json'),
                   '--output-dir', str(root / 'trajectory_memory' / benchmark)]
        run_visible(command, f'{benchmark} trajectory memory')
else:
    print('Memory build is off.')


## 3. Check frozen-baseline compatibility

Run before any GPU evaluation. This checks shared package versions, Python version, development manifests, and ALFWorld YAML against the frozen baseline. If they differ, use a matching runtime or plan a paired baseline rerun before interpreting accuracy.


In [ ]:
from agent_support.runner import package_versions
from agent_support.common import canonical_digest, read_json
frozen = read_json(repo / 'docs' / 'baseline_pilot.json')['runs']
current_versions = package_versions()
for benchmark in BENCHMARKS:
    baseline = frozen[benchmark]
    manifest = read_json(root / 'manifests' / benchmark / 'development.json')
    assert canonical_digest(manifest) == baseline['manifest_sha256'], f'{benchmark} development manifest changed'
    assert len(manifest['tasks']) == 20, f'{benchmark} development manifest is not 20 tasks'
    assert sys.version.split()[0] == baseline['python_version'], 'Python version differs from frozen baseline'
    for package, version in baseline['package_versions'].items():
        assert current_versions.get(package) == version, f'{package} differs: {current_versions.get(package)} vs {version}'
    baseline_dir = root / 'results' / f"{benchmark}_{baseline['run_id']}"
    assert (baseline_dir / 'run.json').exists(), f'Missing frozen baseline directory: {baseline_dir}'
    if benchmark == 'alfworld':
        baseline_meta = read_json(baseline_dir / 'run.json')
        config_file = Path('/content/alfworld_config.yaml')
        snapshot = baseline_dir / 'alfworld_config.yaml'
        assert hashlib.sha256(snapshot.read_bytes()).hexdigest() == baseline_meta['alfworld_config_sha256'], 'Frozen ALFWorld YAML snapshot changed'
        if not config_file.exists() or hashlib.sha256(config_file.read_bytes()).hexdigest() != baseline_meta['alfworld_config_sha256']:
            config_file.write_bytes(snapshot.read_bytes())
        assert hashlib.sha256(config_file.read_bytes()).hexdigest() == baseline_meta['alfworld_config_sha256']
print('Frozen-baseline compatibility checks passed.')


## 4. Smoke test both conditions

Set `RUN_TRAJECTORY_SMOKE=True` in the settings cell, rerun it, then run this cell on a GPU. Each condition gets three development tasks. New run directories are written under `results/`. Inspect `run.json`, `summary.json`, and `episodes/*.json` before a pilot.


In [ ]:
if RUN_TRAJECTORY_SMOKE:
    for benchmark in BENCHMARKS:
        memory_dir = root / 'trajectory_memory' / benchmark
        assert (memory_dir / 'corpus.json').exists() and (memory_dir / 'index.json').exists(), f'Missing memory: {memory_dir}'
        for condition in CONDITIONS:
            suffix = 'expel_retrieval' if condition == 'expel_retrieval_only_v1' else 'm2_trajectory'
            command = [sys.executable, '-m', 'agent_support.runner',
                       '--config', str(repo / 'configs' / f'{benchmark}_{suffix}.json'),
                       '--manifest', str(root / 'manifests' / benchmark / 'development.json'),
                       '--output-root', str(root / 'results'), '--limit', '3']
            run_visible(command, f'{benchmark} {condition} smoke')
else:
    print('Trajectory smoke is off.')


## 5. Run full paired development pilots

After reviewing smoke trajectories, set `RUN_TRAJECTORY_PILOT=True` in the settings cell and run this cell on a GPU. Both conditions use the same memory files and the same 20-task development manifest. Run IDs remain unique.


In [ ]:
if RUN_TRAJECTORY_PILOT:
    assert subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip() == commit
    for benchmark in BENCHMARKS:
        memory_dir = root / 'trajectory_memory' / benchmark
        assert (memory_dir / 'corpus.json').exists() and (memory_dir / 'index.json').exists()
        for condition in CONDITIONS:
            suffix = 'expel_retrieval' if condition == 'expel_retrieval_only_v1' else 'm2_trajectory'
            command = [sys.executable, '-m', 'agent_support.runner',
                       '--config', str(repo / 'configs' / f'{benchmark}_{suffix}.json'),
                       '--manifest', str(root / 'manifests' / benchmark / 'development.json'),
                       '--output-root', str(root / 'results')]
            run_visible(command, f'{benchmark} {condition} full pilot')
else:
    print('Trajectory pilot is off.')


## 6. Compare A, B, and C

After both full conditions finish, set `RUN_TRAJECTORY_COMPARE=True` and run this cell. A is frozen baseline, B is retrieval-only, C is action-conditioned M2. It writes A–B, A–C, and B–C JSON reports to Drive. The comparison refuses incomplete runs, changed manifests, shared-agent setting differences, and different B/C memory artifacts.


In [ ]:
if RUN_TRAJECTORY_COMPARE:
    for benchmark in BENCHMARKS:
        manifest_hash = canonical_digest(read_json(root / 'manifests' / benchmark / 'development.json'))
        runs = {'baseline': root / 'results' / f"{benchmark}_{frozen[benchmark]['run_id']}"}
        for condition in CONDITIONS:
            candidates = []
            for folder in (root / 'results').glob(f'{benchmark}_*'):
                meta_path = folder / 'run.json'
                if not meta_path.exists():
                    continue
                meta = read_json(meta_path)
                if (meta.get('condition') == condition and meta.get('status') == 'complete'
                        and meta.get('limit') is None and meta.get('selected_task_count') == 20
                        and meta.get('selected_manifest_sha256') == manifest_hash
                        and meta.get('git_commit') == commit):
                    candidates.append(folder)
            if not candidates:
                raise RuntimeError(f'Missing complete {benchmark} {condition} pilot from commit {commit}')
            runs[condition] = max(candidates, key=lambda path: path.name)
        comparisons = [('baseline', CONDITIONS[0], 'A_B'),
                       ('baseline', CONDITIONS[1], 'A_C'),
                       (CONDITIONS[0], CONDITIONS[1], 'B_C')]
        for left, right, label in comparisons:
            output = root / 'comparisons' / f'{benchmark}_trajectory_{label}_{commit[:8]}.json'
            command = [sys.executable, '-m', 'agent_support.analysis', str(runs[left]), str(runs[right]),
                       '--trajectory-comparison', '--output', str(output)]
            run_visible(command, f'{benchmark} {label} comparison')
            result = read_json(output)
            print(benchmark, label, result['left']['successes'], 'vs', result['right']['successes'],
                  'helped', result['paired']['helped'], 'hurt', result['paired']['hurt'])
            print('Saved:', output)
else:
    print('Trajectory comparison is off.')
